# 🚀 Domain-Specific Fine-Tuning with OpenShift AI
Welcome to this demonstration of Supervised Fine-Tuning (SFT). In this notebook, we will take a generalized open-source foundation model (Qwen 2.5) and specialize it for a specific industry use-case.

This notebook is based off the work from https://github.com/Red-Hat-AI-Innovation-Team/training_hub/tree/main

**Step 1: Environment Setup**
First, we will install our necessary training dependencies (like `training-hub`) and configure our environment logging. training-hub can tak eosme time to finish 10 min,

In [ ]:
!pip install training-hub[cuda] --no-build-isolation

In [ ]:

# Import training_hub for SFT training
from training_hub import sft

# Standard library imports
import os
import time
import logging
import sys
import shutil
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO

In [ ]:
# Configure logging to show only essential information
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)

# Suppress verbose logging from transformers and other libraries
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("torch").setLevel(logging.WARNING)

print("✅ Logging configured for notebook environment")

In [ ]:
import glob

def find_most_recent_checkpoint(output_dir):
    """
    Find the most recent checkpoint in the training output directory.
    """
    # Get all checkpoint directories under hf_format
    checkpoint_pattern = os.path.join(output_dir, "hf_format", "samples_*")
    checkpoint_dirs = glob.glob(checkpoint_pattern)
    
    if not checkpoint_dirs:
        raise ValueError(f"No checkpoints found in {os.path.join(output_dir, 'hf_format')}")
    
    # Find the most recently created checkpoint
    most_recent_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
    
    return most_recent_checkpoint

print("✅ Checkpoint utility functions defined")

### 📂 Step 2: Dynamic Data Ingestion & Formatting
Below, we have built a **Universal Data Loader**. 

By changing the `ACTIVE_SCENARIO` variable, we will map the raw data from Hugging Face, extract the relevant columns (whether it's tabular data or nested conversation logs), and format it into the exact JSONL structure required for model training. 

*Try changing the scenario to instantly pivot the demo from Cybersecurity, to Medical, to First Aid*

In [ ]:
from datasets import load_dataset
import json

# ==============================================================================
# 🎯 DEMO SCENARIO SELECTOR
# Options: "medical_summary", "cyber_analysis", "first_aid"
# ==============================================================================
ACTIVE_SCENARIO = "cyber_analysis"  # <--- Change this for different datatable for training

SCENARIOS = {
    "medical_summary": {
        "dataset_name": "ccdv/pubmed-summarization",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "article",     
        "target_col": "abstract",   
        "system_prompt": "You are a senior medical researcher. Summarize the following clinical study for a peer review.",
        "user_prefix": "Please summarize this article:\n\n"
    },
    "cyber_analysis": {
        "dataset_name": "Tiamz/cybersecurity-instruction-dataset",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "instruction",  
        "target_col": "answer",      # Fixed from 'output' to 'answer'
        "system_prompt": "You are a Senior Cyber Intrusion Analyst. Provide precise, actionable security incident analysis, mitigation steps, and threat intelligence.",
        "user_prefix": "Security Alert/Inquiry: " 
    },
    "first_aid": {
        "dataset_name": "i-am-mushfiq/FirstAidQA",
        "split": "train",
        "train_size": 1000,
        "eval_start": 1600,
        "input_col": "question", 
        "target_col": "answer",  
        "system_prompt": "You are an emergency first-aid responder. Provide clear, step-by-step, and safe instructions.",
        "user_prefix": "" 
    }
}

CONF = SCENARIOS[ACTIVE_SCENARIO]
print(f"✅ Demo configured for: {ACTIVE_SCENARIO.upper()}")
print(f"📂 Loading {CONF['dataset_name']} (Streaming)...")

raw_dataset = load_dataset(CONF['dataset_name'], split=CONF['split'], streaming=True)

formatted_data = []

print("🔄 Formatting for Training Hub...")
for i, row in enumerate(raw_dataset.take(CONF["train_size"])):
    user_input = f"{CONF['user_prefix']}{row[CONF['input_col']]}"
    expected_output = row[CONF['target_col']]
    
    formatted_data.append({
        "messages": [
            {"role": "system", "content": CONF["system_prompt"]},
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": expected_output},
        ]
    })

# Dynamic paths to keep data completely isolated per industry
DATASET_PATH = f"demo-data/{ACTIVE_SCENARIO}/formatted_training_data.jsonl"
CHECKPOINTS_PATH = f"checkpoints-{ACTIVE_SCENARIO}"

os.makedirs(f"demo-data/{ACTIVE_SCENARIO}", exist_ok=True)

with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for example in formatted_data:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(formatted_data)} records to {DATASET_PATH}")

### ⚙️ Step 3: Hardware-Optimized Configuration
In the next cell, we expose the training hyperparameters. 

We have optimized these settings (such as lowering `MAX_TOKENS_PER_GPU` and `BATCH_SIZE`) to ensure safe, efficient memory management for our specific hardware (L40S GPU). Change these settings to match your hardware.

In [ ]:
################################################################################
# 🤖 Model + Data Paths                                                        #
################################################################################
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_OUTPUT_PATH = "/dev/shm"  

################################################################################
# 🏋️‍♀️ L40S Optimized Training Hyperparameters (Memory Safe)                     #
################################################################################
BATCH_SIZE = 16          # Lowered to reduce memory pressure
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
LR_SCHEDULER = "cosine"
WARMUP_STEPS = 0
SEED = 42

# Performance / Hardware Setup
MAX_TOKENS_PER_GPU = 4096 
MAX_SEQ_LEN = 2048
CHECKPOINT_AT_EPOCH = True
SAVE_FULL_OPTIM_STATE = False

NUM_GPUS = 1
NUM_NODES = 1
NODE_RANK = 0
RDZV_ID = 23
RDZV_ENDPOINT = 'localhost:1738'

from instructlab.training.config import FSDPOptions, ShardingStrategies
# NO_SHARD is best for single GPU memory management
fsdp_opts = FSDPOptions(sharding_strategy=ShardingStrategies.NO_SHARD)

print("⚙️  Training Hyperparameters")
print("=" * 50)
print(f"Base Model: {BASE_MODEL}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Checkpoints Path: {CHECKPOINTS_PATH}")
print(f"Data Output Path: {DATA_OUTPUT_PATH}")
print()
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Number of Epochs: {NUM_EPOCHS}")
print(f"LR Scheduler: {LR_SCHEDULER}")
print(f"Warmup Steps: {WARMUP_STEPS}")
print(f"Seed: {SEED}")
print()
print(f"Max Tokens per GPU: {MAX_TOKENS_PER_GPU:,}")
print(f"Max Sequence Length: {MAX_SEQ_LEN:,}")
print()
print(f"Checkpoint at Epoch: {CHECKPOINT_AT_EPOCH}")
print(f"Save Full Optim State: {SAVE_FULL_OPTIM_STATE}")
print()
print(f"Distributed: {NUM_GPUS} GPUs × {NUM_NODES} nodes = {NUM_GPUS * NUM_NODES} total GPUs")
print(f"Node Rank: {NODE_RANK}")
print(f"RDZV ID: {RDZV_ID}")
print(f"RDZV Endpoint: {RDZV_ENDPOINT}")

In [ ]:
print(f"""Please note that the cell will take a while to run (15-20 minutes on 1xL40S). 
In the meantime, you can check the full training logs in the 
{CHECKPOINTS_PATH}/training_log_node0.log file and see per-step
metrics in {CHECKPOINTS_PATH}/training_metrics_0.jsonl.
\n""")

print("🚀 Starting SFT Training")
print("=" * 60)
print(f"Starting from: {BASE_MODEL}")
print(f"Training data: {DATASET_PATH}")
print(f"Output directory: {CHECKPOINTS_PATH}")
print()

# CLEANUP STEP: Delete old checkpoints for this specific scenario
if os.path.exists(CHECKPOINTS_PATH):
    print(f"🧹 Cleaning up old checkpoints in {CHECKPOINTS_PATH} to save disk space...")
    shutil.rmtree(CHECKPOINTS_PATH)

# Recreate the empty folder for the new run
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

# Capture output to prevent notebook crashes
output_buffer = StringIO()
error_buffer = StringIO()

training_start_time = time.time()

try:
    with redirect_stdout(output_buffer), redirect_stderr(error_buffer):
        # SFT training
        training_result = sft(
            model_path=BASE_MODEL,
            data_path=DATASET_PATH,
            ckpt_output_dir=CHECKPOINTS_PATH,
            num_epochs=NUM_EPOCHS,
            effective_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            max_seq_len=MAX_SEQ_LEN,
            max_tokens_per_gpu=MAX_TOKENS_PER_GPU,
            data_output_dir=DATA_OUTPUT_PATH,
            warmup_steps=WARMUP_STEPS,
            checkpoint_at_epoch=CHECKPOINT_AT_EPOCH,
            accelerate_full_state_at_epoch=SAVE_FULL_OPTIM_STATE,
            nproc_per_node=NUM_GPUS,
            nnodes=NUM_NODES,
            node_rank=NODE_RANK,
            rdzv_id=RDZV_ID,
            rdzv_endpoint=RDZV_ENDPOINT,
            fsdp_options=fsdp_opts,
        )
    
    training_duration = time.time() - training_start_time

    # Find the most recent checkpoint
    final_checkpoint = find_most_recent_checkpoint(CHECKPOINTS_PATH)
    
    print(f"✅ SFT training completed successfully in {training_duration/3600:.2f} hours!")
    print(f"📁 Checkpoint saved to: {final_checkpoint}")
    
except Exception as e:
    print(f"❌ SFT training failed: {e}")
    print("\nError details:")
    print(error_buffer.getvalue())
    raise

### 🔬 Step 5: Model Evaluation (Base vs. Fine-Tuned)
Did the training actually work? To prove it, we will test the models using a "hold-out" set of questions that the model has **never seen before** (zero data leakage).

We will ask the same domain-specific questions to:
1. The original **Base Model** (Generic knowledge)
2. Our new **Fine-Tuned Model** (Specialized behavior)

We will compare answers to the ground truth answer from the datatable.

Watch how the fine-tuned model adopts the precise formatting, tone, and analytical rigor of our chosen industry dataset!

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

NUM_EVAL_SAMPLES = 3 
# 👇 Dynamically pulls the safe start index we set in Cell 4
EVAL_START_INDEX = CONF["eval_start"] 

def run_inference(model, tokenizer, user_text, conf, max_new_tokens=512):
    """Formats the prompt dynamically using the scenario config."""
    messages = [
        {"role": "system", "content": conf["system_prompt"]},
        {"role": "user", "content": f"{conf['user_prefix']}{user_text}"},
    ]
    
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, top_p=0.95)
    
    full_decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if "assistant" in full_decoded:
        return full_decoded.split("assistant")[-1].strip()
    return full_decoded

# --- Load Held-Out Data ---
print(f"📂 Fetching held-out samples from {CONF['dataset_name']}...")
eval_stream = load_dataset(CONF['dataset_name'], split=CONF['split'], streaming=True)

held_out_samples = []
# Skip the exact number of rows we trained on to guarantee zero data leakage
for row in eval_stream.skip(EVAL_START_INDEX).take(NUM_EVAL_SAMPLES):
    held_out_samples.append({
        "input_text": row[CONF["input_col"]],
        "ground_truth": row[CONF["target_col"]]
    })

# --- Evaluate Fine-Tuned Model ---
# (final_checkpoint is defined at the end of Cell 6)
print(f"\n🔄 Loading Fine-Tuned Model...")
ft_tokenizer = AutoTokenizer.from_pretrained(final_checkpoint)
ft_model = AutoModelForCausalLM.from_pretrained(final_checkpoint, torch_dtype=torch.bfloat16, device_map="cuda:0")

ft_answers = [run_inference(ft_model, ft_tokenizer, s["input_text"], CONF) for s in held_out_samples]

# 👇 Critical for your 1x L40S setup: Free up the VRAM before loading the base model
del ft_model
torch.cuda.empty_cache()

# --- Evaluate Base Model ---
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\n🔄 Loading Base Model: {BASE_MODEL_NAME}...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0")

base_answers = [run_inference(base_model, base_tokenizer, s["input_text"], CONF) for s in held_out_samples]

# --- Display Results ---
print("\n" + "=" * 100)
print(f"🔬 EVALUATION RESULTS: {ACTIVE_SCENARIO.replace('_', ' ').upper()}")
print("=" * 100)

for i, sample in enumerate(held_out_samples):
    print(f"\n📄 SAMPLE #{i+1}")
    # Truncate input if it's a long article, show full if it's a short question
    display_input = sample['input_text'][:300] + "..." if len(sample['input_text']) > 300 else sample['input_text']
    print(f"INPUT: {CONF['user_prefix']}{display_input}")
    
    # 👇 Prints the full ground truth without slicing
    print(f"\n📌 GROUND TRUTH:\n{sample['ground_truth']}")
    
    print(f"\n🤖 FINE-TUNED MODEL:\n{ft_answers[i]}")
    print(f"\n📦 BASE MODEL:\n{base_answers[i]}")
    print("-" * 100)